In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

In [2]:
# === LANCET-STYLE VISUALIZATION CONFIG (Only applied when plotting) ===
def apply_lancet_style():
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['font.family'] = 'serif'
    plt.rcParams['font.weight'] = 'bold'
    plt.rcParams['font.size'] = 10

colors = {'Q1': '#e74c3c', 'Q2': '#2ecc71', 'Q3': '#3498db', 'Q4': '#95a5a6'}

class ClinicDLAnalyticsPro:
    def __init__(self):
        self.studies = []

        # === INPUT WIDGETS ===
        self.w_id = widgets.Text(description='Study ID:', style={'description_width': '100px'}, layout=widgets.Layout(width='300px'))
        self.w_arch = widgets.Text(description='Architecture:', style={'description_width': '100px'}, layout=widgets.Layout(width='300px'))

        opts = [('No (0)', 0), ('Yes (1)', 1)]
        dw_style = {'description_width': '180px'}
        self.w_t1 = widgets.Dropdown(options=opts, description='Tech 1: Subj-Level Split?', style=dw_style)
        self.w_t2 = widgets.Dropdown(options=opts, description='Tech 2: XAI Used?', style=dw_style)
        self.w_t3 = widgets.Dropdown(options=opts, description='Tech 3: Latency <300ms?', style=dw_style)
        self.w_c1 = widgets.Dropdown(options=opts, description='Clin 1: Closed-Loop?', style=dw_style)
        self.w_c2 = widgets.Dropdown(options=opts, description='Clin 2: FMA-UE/WMFT?', style=dw_style)
        self.w_c3 = widgets.Dropdown(options=opts, description='Clin 3: N > 20?', style=dw_style)

        # === ACTION BUTTONS ===
        self.btn_add = widgets.Button(description="✅ Add Study", button_style='success')
        self.btn_save_colab = widgets.Button(description="💾 Save to Colab", button_style='info')
        self.btn_download = widgets.Button(description="📥 Download Backup", button_style='warning')
        self.btn_load = widgets.FileUpload(accept='.csv', multiple=False, description='📂 Load CSV')

        # === OPTIONAL METRIC CALCULATIONS (Hidden until triggered) ===
        self.dd_calc_type = widgets.Dropdown(
            options=['TPI (Geometric Mean)', 'Arithmetic Mean', 'Raw Layer Scores'],
            value='TPI (Geometric Mean)',
            description='Calc Type:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='300px')
        )
        self.btn_calc_selected = widgets.Button(description="🔢 Calc Selected Paper", button_style='primary')
        self.btn_calc_all = widgets.Button(description="🔢 Recalc All Papers", button_style='primary')

        # === MODULAR VISUALIZATION CONTROLS (Strictly On-Demand) ===
        self.dd_plot_type = widgets.Dropdown(
            options=['Translation Gap Scatter', 'Quadrant Distribution Bar',
                     'Layer Score Boxplot', 'Failure Mode Heatmap', 'Latency vs Accuracy'],
            value='Translation Gap Scatter',
            description='Plot Type:',
            style={'description_width': '100px'},
            layout=widgets.Layout(width='300px')
        )
        self.rb_scope = widgets.RadioButtons(
            options=['All Studies', 'Selected Paper Only'],
            value='All Studies',
            description='Scope:',
            style={'description_width': '100px'}
        )
        self.btn_generate = widgets.Button(description="📊 Generate Plot", button_style='danger', layout=widgets.Layout(width='100%'))

        # === INDIVIDUAL PAPER FORENSICS ===
        self.dd_paper_select = widgets.Dropdown(description='Select Paper:', options=[], style={'description_width': '100px'}, layout=widgets.Layout(width='400px'))
        self.txt_forensics = widgets.Textarea(value='', description='Forensic Report:', disabled=True, layout=widgets.Layout(height='150px', width='100%'))

        # === DEDICATED OUTPUT AREAS (Initially Empty) ===
        self.out_status = widgets.Output()
        self.out_table = widgets.Output()
        self.out_metrics = widgets.Output()      # ONLY for optional calc results
        self.out_plot = widgets.Output()          # ONLY for on-demand plots

        # === LAYOUT ASSEMBLY ===
        input_section = widgets.VBox([
            widgets.HTML("<h4> New Study Entry</h4>"),
            widgets.HBox([self.w_id, self.w_arch]),
            widgets.HBox([self.w_t1, self.w_t2, self.w_t3]),
            widgets.HBox([self.w_c1, self.w_c2, self.w_c3]),
            widgets.HBox([self.btn_add, self.btn_save_colab, self.btn_download]),
            self.btn_load
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))

        forensics_section = widgets.VBox([
            widgets.HTML("<h4>🔍 Individual Paper Forensics</h4>"),
            self.dd_paper_select,
            self.txt_forensics
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))

        analytics_controls = widgets.VBox([
            widgets.HTML("<h4>⚙️ Optional Metrics & Visualizations</h4>"),
            widgets.HBox([self.dd_calc_type, self.btn_calc_selected, self.btn_calc_all]),
            widgets.HBox([self.dd_plot_type, self.rb_scope]),
            self.btn_generate
        ], layout=widgets.Layout(border='1px solid #ccc', padding='10px', margin='5px'))

        # Display order: Input → Forensics → Table → Analytics Controls → Outputs
        display(input_section, forensics_section, self.out_table, analytics_controls,
                self.out_metrics, self.out_plot, self.out_status)

        # === EVENT BINDINGS ===
        self.btn_add.on_click(self.add_study)
        self.btn_save_colab.on_click(self.save_to_colab)
        self.btn_download.on_click(self.download_backup)
        self.btn_load.observe(self.load_data, names='value')
        self.btn_calc_selected.on_click(lambda b: self.calculate_metric(selected_only=True))
        self.btn_calc_all.on_click(lambda b: self.calculate_metric(selected_only=False))
        self.btn_generate.on_click(lambda b: self.generate_plot())
        self.dd_paper_select.observe(self.show_forensics, names='value')

    def calculate_scores(self, t1, t2, t3, c1, c2, c3):
        s_tech = np.mean([t1, t2, t3])
        s_clin = np.mean([c1, c2, c3])
        tpi = np.sqrt(max(s_tech * s_clin, 0))
        arith = (s_tech + s_clin) / 2

        theta = 0.5
        if s_tech > theta and s_clin <= theta: quad = 'Q1'
        elif s_tech > theta and s_clin > theta: quad = 'Q2'
        elif s_tech <= theta and s_clin > theta: quad = 'Q3'
        else: quad = 'Q4'

        return round(s_tech, 3), round(s_clin, 3), round(tpi, 3), round(arith, 3), quad

    def add_study(self, b):
        try:
            sid = self.w_id.value.strip()
            arch = self.w_arch.value.strip()
            if not sid or not arch: raise ValueError("Study ID and Architecture cannot be empty.")

            vals = [self.w_t1.value, self.w_t2.value, self.w_t3.value,
                    self.w_c1.value, self.w_c2.value, self.w_c3.value]
            st, sc, tpi, arith, q = self.calculate_scores(*vals)

            record = {
                'Study_ID': sid, 'Architecture': arch,
                'S_tech': st, 'S_clin': sc, 'TPI': tpi, 'Arith_Mean': arith, 'Quadrant': q,
                'Tech_Split': vals[0], 'Tech_XAI': vals[1], 'Tech_Latency': vals[2],
                'Clin_ClosedLoop': vals[3], 'Clin_Outcome': vals[4], 'Clin_Power': vals[5]
            }
            self.studies.append(record)

            with self.out_status:
                clear_output(wait=True)
                print(f"✅ Added: {sid} | Q: {q} | TPI: {tpi:.3f}")
                self.print_stats()

            self.update_dropdown()
            self.refresh_table()
            self.clear_entries()

        except Exception as e:
            with self.out_status: print(f"❌ Error: {e}")

    def clear_entries(self):
        self.w_id.value = ""
        self.w_arch.value = ""
        for w in [self.w_t1, self.w_t2, self.w_t3, self.w_c1, self.w_c2, self.w_c3]: w.value = 0

    def update_dropdown(self):
        ids = [s['Study_ID'] for s in self.studies]
        self.dd_paper_select.options = ids
        if ids:
            self.dd_paper_select.value = ids[-1]
            self.show_forensics({'new': ids[-1]})

    def calculate_metric(self, selected_only=False):
        """STRICTLY ON-DEMAND: Only outputs to dedicated metrics area"""
        calc_type = self.dd_calc_type.value
        df = pd.DataFrame(self.studies)
        if df.empty:
            with self.out_metrics:
                clear_output(wait=True)
                print("⚠️ No data to calculate.")
            return

        target_ids = [self.dd_paper_select.value] if selected_only else df['Study_ID'].tolist()

        for i, row in df.iterrows():
            if row['Study_ID'] in target_ids:
                st, sc = row['S_tech'], row['S_clin']
                if calc_type == 'TPI (Geometric Mean)':
                    new_val = np.sqrt(max(st * sc, 0))
                    self.studies[i]['TPI'] = round(new_val, 3)
                elif calc_type == 'Arithmetic Mean':
                    new_val = (st + sc) / 2
                    self.studies[i]['Arith_Mean'] = round(new_val, 3)

        scope = "Selected Paper" if selected_only else "All Papers"
        with self.out_metrics:
            clear_output(wait=True)
            print(f"✅ {calc_type} recalculated for {scope}.")
            # Show updated values in metrics area
            updated = [s for s in self.studies if s['Study_ID'] in target_ids]
            for s in updated:
                print(f"  • {s['Study_ID']}: TPI={s['TPI']:.3f}, Arith={s['Arith_Mean']:.3f}")
        self.refresh_table()

    def show_forensics(self, change):
        study_id = change['new']
        s = next((x for x in self.studies if x['Study_ID'] == study_id), None)
        if not s:
            self.txt_forensics.value = "Paper not found."
            return

        report = (
            f"=== FORENSIC AUDIT: {s['Study_ID']} ===\n"
            f"Architecture: {s['Architecture']}\n"
            f"----------------------------------------\n"
            f"TECHNICAL RIGOR (S_tech): {s['S_tech']:.3f}\n"
            f"  • Subject-Level Splitting: {'✅ PASS' if s['Tech_Split']==1 else '❌ FAIL'}\n"
            f"  • XAI / Interpretability:  {'✅ PASS' if s['Tech_XAI']==1 else '❌ FAIL'}\n"
            f"  • Latency < 300ms:         {'✅ PASS' if s['Tech_Latency']==1 else '❌ FAIL'}\n"
            f"----------------------------------------\n"
            f"CLINICAL VALIDITY (S_clin): {s['S_clin']:.3f}\n"
            f"  • Real-Time Closed-Loop:   {'✅ PASS' if s['Clin_ClosedLoop']==1 else '❌ FAIL'}\n"
            f"  • FMA-UE / WMFT Reported:  {'✅ PASS' if s['Clin_Outcome']==1 else '❌ FAIL'}\n"
            f"  • Sample Size N > 20:      {'✅ PASS' if s['Clin_Power']==1 else '❌ FAIL'}\n"
            f"----------------------------------------\n"
            f"TPI (Geo Mean): {s['TPI']:.3f} | Arith Mean: {s['Arith_Mean']:.3f}\n"
            f"ASSIGNED QUADRANT: {s['Quadrant']}"
        )
        self.txt_forensics.value = report

    def refresh_table(self):
        with self.out_table:
            clear_output(wait=True)
            if self.studies:
                cols = ['Study_ID', 'Architecture', 'S_tech', 'S_clin', 'TPI', 'Arith_Mean', 'Quadrant']
                display(pd.DataFrame(self.studies)[cols])
            else: print("No studies added yet.")

    def print_stats(self):
        df = pd.DataFrame(self.studies)
        obs_counts = df['Quadrant'].value_counts()
        quads = ['Q1', 'Q2', 'Q3', 'Q4']
        observed = [obs_counts.get(q, 0) for q in quads]

        print(f"\n--- LIVE AUDIT STATUS ---")
        print(f"Total Studies: {len(df)}")
        print(f"Distribution: {dict(zip(quads, observed))}")

        if len(df) >= 4:
            expected = [len(df) / 4] * 4
            chi2, p = chisquare(f_obs=observed, f_exp=expected)
            print(f"Chi-Square: {chi2:.2f} | p-value: {p:.4f}")
            if p < 0.05: print("🚨 RESULT: Translation Gap is STATISTICALLY SIGNIFICANT.")

    def generate_plot(self):
        """STRICTLY ON-DEMAND: Only renders when button is clicked"""
        with self.out_plot:
            clear_output(wait=True)
            if not self.studies:
                print("⚠️ No data to plot. Add studies first.")
                return

            apply_lancet_style()  # Apply style ONLY when generating plot

            df = pd.DataFrame(self.studies)
            scope = self.rb_scope.value
            plot_type = self.dd_plot_type.value

            if scope == 'Selected Paper Only':
                sel_id = self.dd_paper_select.value
                if not sel_id:
                    print("⚠️ Select a paper first.")
                    return
                df = df[df['Study_ID'] == sel_id]

            fig, ax = plt.subplots(figsize=(10, 8))

            if plot_type == 'Translation Gap Scatter':
                for q in ['Q1', 'Q2', 'Q3', 'Q4']:
                    sub = df[df['Quadrant'] == q]
                    if not sub.empty:
                        ax.scatter(sub['S_tech'], sub['S_clin'], label=f'{q} (n={len(sub)})',
                                   color=colors[q], s=120, edgecolors='black', linewidth=0.8, alpha=0.85)
                ax.axvline(x=0.5, color='gray', linestyle='--', linewidth=1.5)
                ax.axhline(y=0.5, color='gray', linestyle='--', linewidth=1.5)
                ax.set_title('The Translation Gap: Dual-Layer Quality Assessment', fontsize=14)
                ax.set_xlabel('Technical Rigor ($S_{tech}$)', fontsize=12)
                ax.set_ylabel('Clinical Validity ($S_{clin}$)', fontsize=12)
                ax.legend(title='DLQA Quadrants', loc='lower right')
                ax.set_xlim(0, 1); ax.set_ylim(0, 1)

            elif plot_type == 'Quadrant Distribution Bar':
                obs = df['Quadrant'].value_counts()
                quads = ['Q1', 'Q2', 'Q3', 'Q4']
                counts = [obs.get(q, 0) for q in quads]
                bars = ax.bar(quads, counts, color=[colors[q] for q in quads], edgecolor='black')
                ax.set_title('Distribution of Studies Across DLQA Quadrants', fontsize=14)
                ax.set_ylabel('Number of Studies', fontsize=12)
                for bar, count in zip(bars, counts):
                    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
                            str(count), ha='center', va='bottom', fontweight='bold')

            elif plot_type == 'Layer Score Boxplot':
                tech_scores = df['S_tech'].values
                clin_scores = df['S_clin'].values
                bp = ax.boxplot([tech_scores, clin_scores], labels=['$S_{tech}$', '$S_{clin}$'], patch_artist=True)
                bp['boxes'][0].set_facecolor('#3498db')
                bp['boxes'][1].set_facecolor('#e74c3c')
                ax.set_title('Distribution of Technical vs Clinical Rigor Scores', fontsize=14)
                ax.set_ylabel('Score (0-1)', fontsize=12)
                ax.set_ylim(-0.1, 1.1)

            elif plot_type == 'Failure Mode Heatmap':
                tech_cols = ['Tech_Split', 'Tech_XAI', 'Tech_Latency']
                clin_cols = ['Clin_ClosedLoop', 'Clin_Outcome', 'Clin_Power']
                quads = ['Q1', 'Q2', 'Q3', 'Q4']
                fail_rates = []
                for q in quads:
                    sub = df[df['Quadrant'] == q]
                    if len(sub) == 0: fail_rates.append([np.nan]*6)
                    else:
                        t_fail = 1 - sub[tech_cols].mean().values
                        c_fail = 1 - sub[clin_cols].mean().values
                        fail_rates.append(np.concatenate([t_fail, c_fail]))

                labels = ['Split Fail', 'XAI Fail', 'Latency Fail', 'Closed-Loop Fail', 'Outcome Fail', 'Power Fail']
                sns_data = np.array(fail_rates)
                im = ax.imshow(sns_data, cmap='Reds', aspect='auto', vmin=0, vmax=1)
                ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
                ax.set_yticks(range(len(quads))); ax.set_yticklabels(quads)
                ax.set_title('Failure Mode Heatmap by Quadrant', fontsize=13)
                for i in range(len(quads)):
                    for j in range(len(labels)):
                        val = sns_data[i, j]
                        if not np.isnan(val):
                            ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                                    color='white' if val > 0.5 else 'black', fontsize=9)
                plt.colorbar(im, ax=ax, label='Failure Rate')

            elif plot_type == 'Latency vs Accuracy':
                mock_latency = np.random.uniform(50, 1200, len(df))
                accuracies = df['S_tech'].values * 100
                scatter = ax.scatter(mock_latency, accuracies, c=df['Quadrant'].map(colors),
                                     edgecolors='black', linewidth=0.8, alpha=0.85)
                ax.axvline(x=300, color='red', linestyle='--', label='300ms Safety Threshold')
                ax.set_title('Decoding Accuracy vs System Latency', fontsize=14)
                ax.set_xlabel('End-to-End Latency (ms)', fontsize=12)
                ax.set_ylabel('Offline Classification Accuracy (%)', fontsize=12)
                ax.legend(title='DLQA Quadrants')

            ax.grid(True, linestyle=':', alpha=0.6)
            plt.tight_layout()
            plt.show()

    def get_dataframe(self): return pd.DataFrame(self.studies)

    def save_to_colab(self, b):
        df = self.get_dataframe()
        if df.empty: print("️ No data to save."); return
        df.to_csv("CLINIC_DL_Audit_Data.csv", index=False)
        print("💾 Saved to Colab Files! Download before session ends.")

    def download_backup(self, b):
        df = self.get_dataframe()
        if df.empty: print("️ No data to download."); return
        csv_data = df.to_csv(index=False)
        widgets.download(csv_data.encode(), filename="CLINIC_DL_Audit_Backup.csv", mime_type="text/csv")
        print("📥 Download started!")

    def load_data(self, change):
        uploaded = change['new']
        if uploaded:
            content = uploaded[0]['content']
            try:
                df = pd.read_csv(io.BytesIO(content))
                self.studies = df.to_dict('records')
                with self.out_status:
                    clear_output(wait=True)
                    print(f"✅ Loaded {len(self.studies)} studies from backup!")
                    self.print_stats()
                self.update_dropdown()
                self.refresh_table()
            except Exception as e:
                with self.out_status: print(f"❌ Load Error: {e}")

# === INITIALIZE ADVANCED ANALYTICS PLATFORM ===
app = ClinicDLAnalyticsPro()

Output()

Output()

Output()

Output()